# WCC Experiments: Gurobi vs Nearest-Neighbour Heuristic

Run the MP-CVRP-P model on multiple instances:
- Small instance (10 cafes)
- Medium instance (15 cafes)
- Full instance (34 cafes)

Then run the nearest-neighbour heuristic on the same instances and compare.

In [ ]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import networkx as nx
import time as time_module
import matplotlib.pyplot as plt

DATA_DIR = '../data/merged'  # adjust to your repo structure

## 1. Load Data

In [ ]:
# Products
products = pd.read_csv(f'{DATA_DIR}/products.csv')
P = products['product_id'].tolist()
revenue = dict(zip(products['product_id'], products['revenue_per_unit']))
weight = dict(zip(products['product_id'], products['weight_per_unit_kg']))

# Cafes
cafes = pd.read_csv(f'{DATA_DIR}/cafes.csv')
all_cafe_ids = cafes['cafe_id'].tolist()
n_all = len(all_cafe_ids)

# Demands
demands_df = pd.read_csv(f'{DATA_DIR}/demands.csv')

# Distance and time matrices
dist_df = pd.read_csv(f'{DATA_DIR}/distance_matrix_km.csv', index_col=0)
time_df = pd.read_csv(f'{DATA_DIR}/time_matrix_min.csv', index_col=0)

# Vans
vans_df = pd.read_csv(f'{DATA_DIR}/vans.csv')

# Perishability
perish_df = pd.read_csv(f'{DATA_DIR}/perishability_params.csv')
perish_params = dict(zip(perish_df['parameter'], perish_df['value']))

print(f'Loaded: {n_all} cafes, {len(P)} products, {len(vans_df)} vans available')

## 2. Instance Builder

Function that builds all the Gurobi-ready data structures for a given subset of cafes.

In [ ]:
def build_instance(n_cafes, num_vans, t_max_hours=3.0):
    """
    Build instance data for a given number of cafes and vans.
    Uses the first n_cafes cafes from the dataset.
    
    Returns a dict with all sets, parameters, and data needed for the model.
    """
    # Select cafes (first n_cafes)
    cafe_ids = all_cafe_ids[:n_cafes]
    
    # Node mapping: 0 = depot, 1..n = cafes
    n = len(cafe_ids)
    V = list(range(n + 1))
    V_c = list(range(1, n + 1))
    cafe_to_node = {cid: i + 1 for i, cid in enumerate(cafe_ids)}
    node_to_cafe = {i + 1: cid for i, cid in enumerate(cafe_ids)}
    
    # Vans
    K = list(range(num_vans))
    Q = {k: vans_df.loc[k, 'capacity_kg'] for k in K}
    
    # Demand: d[i][p]
    d = {}
    for _, row in demands_df.iterrows():
        if row['cafe_id'] in cafe_to_node:
            node = cafe_to_node[row['cafe_id']]
            if node not in d:
                d[node] = {}
            d[node][row['product_id']] = row['daily_demand']
    
    # Node labels for matrix indexing
    node_labels = ['depot'] + cafe_ids
    
    # Distance, cost, time
    avg_fuel = np.mean([vans_df.loc[k, 'fuel_cost_per_km'] for k in K])
    c = {}      # travel cost
    dist = {}   # distance km
    t = {}      # travel time in hours
    A = []      # arcs
    
    for i in V:
        for j in V:
            if i != j:
                di = dist_df.loc[node_labels[i], node_labels[j]]
                ti = time_df.loc[node_labels[i], node_labels[j]]
                c[i, j] = round(di * avg_fuel, 4)
                dist[i, j] = round(di, 4)
                t[i, j] = round(ti / 60, 4)  # min to hours
                A.append((i, j))
    
    service_time = perish_params['service_time_minutes'] / 60  # hours
    
    return {
        'n': n, 'V': V, 'V_c': V_c, 'K': K, 'P': P, 'A': A,
        'd': d, 'c': c, 'dist': dist, 't': t, 'Q': Q,
        'revenue': revenue, 'weight': weight,
        'T_max': t_max_hours, 'service_time': service_time,
        'node_to_cafe': node_to_cafe, 'cafe_ids': cafe_ids
    }

## 3. Gurobi Model (MP-CVRP-P with MTZ)

Full multi-product CVRP with perishability constraints and MTZ subtour elimination.

In [ ]:
def solve_gurobi(inst, time_limit=300, verbose=False):
    """
    Solve the MP-CVRP-P using Gurobi with MTZ subtour elimination.
    
    Returns a dict with: objective, revenue, travel_cost, distance,
    solve_time, mip_gap, routes, status.
    """
    n = inst['n']
    V = inst['V']
    V_c = inst['V_c']
    K = inst['K']
    P = inst['P']
    A = inst['A']
    d = inst['d']
    c = inst['c']
    dist = inst['dist']
    t = inst['t']
    Q = inst['Q']
    rev = inst['revenue']
    w = inst['weight']
    T_max = inst['T_max']
    s = inst['service_time']
    
    m = gp.Model('MP_CVRP_P')
    if not verbose:
        m.Params.OutputFlag = 0
    m.Params.TimeLimit = time_limit
    
    # --- Decision Variables ---
    # x[i,j,k] = 1 if van k travels from node i to node j
    x = m.addVars(A, K, vtype=GRB.BINARY, name='x')
    
    # q[i,p,k] = quantity of product p delivered to cafe i by van k
    q = m.addVars(V_c, P, K, vtype=GRB.CONTINUOUS, lb=0, name='q')
    
    # u[i,k] = position of node i in van k's route (MTZ)
    u = m.addVars(V_c, K, vtype=GRB.CONTINUOUS, lb=1, ub=n, name='u')
    
    # --- Objective: maximise revenue - travel cost ---
    total_revenue = gp.quicksum(
        rev[p] * q[i, p, k]
        for i in V_c for p in P for k in K
    )
    total_travel_cost = gp.quicksum(
        c[i, j] * x[i, j, k]
        for (i, j) in A for k in K
    )
    m.setObjective(total_revenue - total_travel_cost, GRB.MAXIMIZE)
    
    # --- Constraints ---
    
    # 6.1 Visit: each cafe visited by exactly one van
    m.addConstrs((
        gp.quicksum(x[i, j, k] for j in V if j != i for k in K) == 1
        for i in V_c
    ), name='visit')
    
    # 6.2 Depot flow: each van leaves depot once
    m.addConstrs((
        gp.quicksum(x[0, j, k] for j in V_c) == 1
        for k in K
    ), name='leave_depot')
    
    # 6.2 Depot flow: each van returns to depot once
    m.addConstrs((
        gp.quicksum(x[i, 0, k] for i in V_c) == 1
        for k in K
    ), name='return_depot')
    
    # 6.3 Flow conservation: if van arrives, it must leave
    m.addConstrs((
        gp.quicksum(x[i, j, k] for i in V if i != j)
        == gp.quicksum(x[j, i, k] for i in V if i != j)
        for j in V_c for k in K
    ), name='flow')
    
    # 6.4 Demand satisfaction: total delivered = demand
    m.addConstrs((
        gp.quicksum(q[i, p, k] for k in K) == d[i][p]
        for i in V_c for p in P
    ), name='demand')
    
    # 6.5 Linking: can only deliver if van visits cafe
    m.addConstrs((
        q[i, p, k] <= d[i][p] * gp.quicksum(x[i, j, k] for j in V if j != i)
        for i in V_c for p in P for k in K
    ), name='link')
    
    # 6.6 Van capacity
    m.addConstrs((
        gp.quicksum(w[p] * q[i, p, k] for i in V_c for p in P) <= Q[k]
        for k in K
    ), name='capacity')
    
    # 6.7 MTZ subtour elimination
    m.addConstrs((
        u[i, k] - u[j, k] + n * x[i, j, k] <= n - 1
        for i in V_c for j in V_c if i != j for k in K
    ), name='mtz')
    
    # 6.8 Perishability: route duration <= T_max (applied to all vans)
    m.addConstrs((
        gp.quicksum(t[i, j] * x[i, j, k] for (i, j) in A)
        + s * gp.quicksum(x[i, j, k] for i in V_c for j in V if j != i)
        <= T_max
        for k in K
    ), name='perishability')
    
    # --- Solve ---
    start = time_module.time()
    m.optimize()
    solve_time = time_module.time() - start
    
    # --- Extract results ---
    result = {
        'n_cafes': n,
        'n_vans': len(K),
        'T_max': T_max,
        'status': m.Status,
        'solve_time': round(solve_time, 2),
    }
    
    if m.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT] and m.SolCount > 0:
        result['objective'] = round(m.ObjVal, 2)
        result['mip_gap'] = round(m.MIPGap * 100, 2)  # percentage
        
        # Calculate revenue and cost separately
        tot_rev = sum(rev[p] * q[i, p, k].X for i in V_c for p in P for k in K)
        tot_cost = sum(c[i, j] * x[i, j, k].X for (i, j) in A for k in K)
        tot_dist = sum(dist[i, j] * x[i, j, k].X for (i, j) in A for k in K)
        result['revenue'] = round(tot_rev, 2)
        result['travel_cost'] = round(tot_cost, 2)
        result['total_distance_km'] = round(tot_dist, 2)
        
        # Extract routes
        routes = {}
        for k in K:
            route = []
            current = 0  # start at depot
            while True:
                next_node = None
                for j in V:
                    if j != current and x[current, j, k].X > 0.5:
                        next_node = j
                        break
                if next_node is None or next_node == 0:
                    break
                route.append(next_node)
                current = next_node
            routes[k] = route
        result['routes'] = routes
    else:
        result['objective'] = None
        result['mip_gap'] = None
        result['revenue'] = None
        result['travel_cost'] = None
        result['total_distance_km'] = None
        result['routes'] = {}
    
    return result

## 4. Nearest-Neighbour Heuristic

In [ ]:
def solve_heuristic(inst):
    """
    Nearest-neighbour construction heuristic for the MP-CVRP-P.
    
    For each van:
      1. Start at depot
      2. Find nearest unvisited cafe whose total demand weight fits in remaining capacity
         AND whose addition doesn't exceed T_max route duration
      3. Visit it, update remaining capacity and route time
      4. Repeat until no more cafes fit
      5. Return to depot
    
    Returns a dict with same structure as solve_gurobi for comparison.
    """
    n = inst['n']
    V_c = inst['V_c']
    K = inst['K']
    P = inst['P']
    d = inst['d']
    c = inst['c']
    dist_data = inst['dist']
    t = inst['t']
    Q = inst['Q']
    rev = inst['revenue']
    w = inst['weight']
    T_max = inst['T_max']
    s = inst['service_time']
    
    start = time_module.time()
    
    # Precompute total weight per cafe
    cafe_weight = {}
    for i in V_c:
        cafe_weight[i] = sum(w[p] * d[i][p] for p in P)
    
    # Precompute total revenue per cafe
    cafe_rev = {}
    for i in V_c:
        cafe_rev[i] = sum(rev[p] * d[i][p] for p in P)
    
    unvisited = set(V_c)
    routes = {}
    total_dist = 0
    total_cost = 0
    total_revenue = 0
    
    for k in K:
        route = []
        remaining_cap = Q[k]
        route_time = 0  # hours
        current = 0  # depot
        
        while unvisited:
            # Find nearest feasible unvisited cafe
            best_cafe = None
            best_dist = float('inf')
            
            for i in unvisited:
                # Check capacity
                if cafe_weight[i] > remaining_cap:
                    continue
                
                # Check time: travel to cafe + service + return to depot
                time_to_cafe = t[current, i]
                time_back = t[i, 0]
                new_route_time = route_time + time_to_cafe + s
                if new_route_time + time_back > T_max:
                    continue
                
                # Nearest by distance
                if dist_data[current, i] < best_dist:
                    best_dist = dist_data[current, i]
                    best_cafe = i
            
            if best_cafe is None:
                break
            
            # Visit the cafe
            route.append(best_cafe)
            unvisited.remove(best_cafe)
            total_dist += dist_data[current, best_cafe]
            total_cost += c[current, best_cafe]
            total_revenue += cafe_rev[best_cafe]
            remaining_cap -= cafe_weight[best_cafe]
            route_time += t[current, best_cafe] + s
            current = best_cafe
        
        # Return to depot
        if route:
            total_dist += dist_data[current, 0]
            total_cost += c[current, 0]
        
        routes[k] = route
    
    solve_time = time_module.time() - start
    
    # Check for unserved cafes
    served = sum(len(r) for r in routes.values())
    
    return {
        'n_cafes': n,
        'n_vans': len(K),
        'T_max': T_max,
        'status': 'COMPLETE' if not unvisited else f'PARTIAL ({len(unvisited)} unserved)',
        'solve_time': round(solve_time, 4),
        'objective': round(total_revenue - total_cost, 2),
        'mip_gap': 0,
        'revenue': round(total_revenue, 2),
        'travel_cost': round(total_cost, 2),
        'total_distance_km': round(total_dist, 2),
        'routes': routes,
        'cafes_served': served,
        'cafes_unserved': len(unvisited)
    }

## 5. Helper: Print Results

In [ ]:
def print_result(label, res, inst=None):
    """Pretty print a result dict."""
    print(f'\n{"="*60}')
    print(f'{label}')
    print(f'{"="*60}')
    print(f'Status:         {res["status"]}')
    print(f'Solve time:     {res["solve_time"]}s')
    print(f'Objective:      ${res["objective"]}')
    print(f'Revenue:        ${res["revenue"]}')
    print(f'Travel cost:    ${res["travel_cost"]}')
    print(f'Total distance: {res["total_distance_km"]} km')
    if res.get('mip_gap') is not None:
        print(f'MIP gap:        {res["mip_gap"]}%')
    if 'cafes_unserved' in res and res['cafes_unserved'] > 0:
        print(f'WARNING: {res["cafes_unserved"]} cafes not served!')
    
    print(f'\nRoutes:')
    for k, route in res['routes'].items():
        if inst:
            names = [inst['node_to_cafe'].get(i, f'node_{i}') for i in route]
            print(f'  Van {k}: depot -> {" -> ".join(names)} -> depot ({len(route)} cafes)')
        else:
            print(f'  Van {k}: depot -> {route} -> depot ({len(route)} cafes)')

## 6. Run Experiments

### 6.1 Small Instance (10 cafes, 2 vans)

In [ ]:
# Small instance: 10 cafes, 2 vans
inst_small = build_instance(n_cafes=10, num_vans=2, t_max_hours=3.0)

res_gurobi_small = solve_gurobi(inst_small, time_limit=120, verbose=False)
print_result('GUROBI - Small (10 cafes, 2 vans)', res_gurobi_small, inst_small)

res_heur_small = solve_heuristic(inst_small)
print_result('HEURISTIC - Small (10 cafes, 2 vans)', res_heur_small, inst_small)

### 6.2 Medium Instance (15 cafes, 3 vans)

In [ ]:
# Medium instance: 15 cafes, 3 vans
inst_med = build_instance(n_cafes=15, num_vans=3, t_max_hours=3.0)

res_gurobi_med = solve_gurobi(inst_med, time_limit=300, verbose=False)
print_result('GUROBI - Medium (15 cafes, 3 vans)', res_gurobi_med, inst_med)

res_heur_med = solve_heuristic(inst_med)
print_result('HEURISTIC - Medium (15 cafes, 3 vans)', res_heur_med, inst_med)

### 6.3 Full Instance (34 cafes, 3 vans)

In [ ]:
# Full instance: 34 cafes, 3 vans
inst_full = build_instance(n_cafes=34, num_vans=3, t_max_hours=3.0)

res_gurobi_full = solve_gurobi(inst_full, time_limit=300, verbose=False)
print_result('GUROBI - Full (34 cafes, 3 vans)', res_gurobi_full, inst_full)

res_heur_full = solve_heuristic(inst_full)
print_result('HEURISTIC - Full (34 cafes, 3 vans)', res_heur_full, inst_full)

### 6.4 Full Instance (34 cafes, 5 vans)

In [ ]:
# Full instance with more vans: 34 cafes, 5 vans
inst_full5 = build_instance(n_cafes=34, num_vans=5, t_max_hours=3.0)

res_gurobi_full5 = solve_gurobi(inst_full5, time_limit=300, verbose=False)
print_result('GUROBI - Full (34 cafes, 5 vans)', res_gurobi_full5, inst_full5)

res_heur_full5 = solve_heuristic(inst_full5)
print_result('HEURISTIC - Full (34 cafes, 5 vans)', res_heur_full5, inst_full5)

## 7. Comparison Summary

In [ ]:
# Build comparison table
experiments = [
    ('Small (10c, 2v)', res_gurobi_small, res_heur_small),
    ('Medium (15c, 3v)', res_gurobi_med, res_heur_med),
    ('Full (34c, 3v)', res_gurobi_full, res_heur_full),
    ('Full (34c, 5v)', res_gurobi_full5, res_heur_full5),
]

rows = []
for name, gur, heur in experiments:
    if gur['objective'] is not None and heur['objective'] is not None:
        improvement = ((gur['objective'] - heur['objective']) / abs(heur['objective'])) * 100
    else:
        improvement = None
    
    rows.append({
        'Instance': name,
        'Gurobi Profit ($)': gur['objective'],
        'Gurobi Time (s)': gur['solve_time'],
        'Gurobi Gap (%)': gur.get('mip_gap'),
        'Gurobi Dist (km)': gur['total_distance_km'],
        'Heuristic Profit ($)': heur['objective'],
        'Heuristic Time (s)': heur['solve_time'],
        'Heuristic Dist (km)': heur['total_distance_km'],
        'Improvement (%)': round(improvement, 1) if improvement else None
    })

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

In [ ]:
# Bar chart: Gurobi vs Heuristic profit
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = [name for name, _, _ in experiments]
gurobi_profits = [g['objective'] if g['objective'] else 0 for _, g, _ in experiments]
heur_profits = [h['objective'] if h['objective'] else 0 for _, _, h in experiments]

x_pos = np.arange(len(labels))
width = 0.35

# Profit comparison
axes[0].bar(x_pos - width/2, gurobi_profits, width, label='Gurobi', color='#1F4E79')
axes[0].bar(x_pos + width/2, heur_profits, width, label='Heuristic', color='#2E75B6')
axes[0].set_ylabel('Profit ($)')
axes[0].set_title('Profit: Gurobi vs Heuristic')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(labels, rotation=15, ha='right')
axes[0].legend()

# Distance comparison
gurobi_dists = [g['total_distance_km'] if g['total_distance_km'] else 0 for _, g, _ in experiments]
heur_dists = [h['total_distance_km'] if h['total_distance_km'] else 0 for _, _, h in experiments]

axes[1].bar(x_pos - width/2, gurobi_dists, width, label='Gurobi', color='#1F4E79')
axes[1].bar(x_pos + width/2, heur_dists, width, label='Heuristic', color='#2E75B6')
axes[1].set_ylabel('Distance (km)')
axes[1].set_title('Total Distance: Gurobi vs Heuristic')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(labels, rotation=15, ha='right')
axes[1].legend()

plt.tight_layout()
plt.savefig('gurobi_vs_heuristic.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to gurobi_vs_heuristic.png')

In [ ]:
# Save results to CSV
comparison.to_csv('experiment_results.csv', index=False)
print('Results saved to experiment_results.csv')